# M01 — Map the AI/ML Landscape

**Whole first:** `data → training → model state → inference → application`.

Surround that path with **retrieval/RAG, tools/agents, memory, evaluation/observability, and compute/infrastructure**. Track both **data flow** (values moving) and **control flow** (what decides the next action).

Structured data has explicit fields/schema; unstructured data includes free text, images, audio, and documents. One record can contain both.

Model-family orientation: **classical ML** includes linear/tree/probabilistic/neighbor methods; **neural networks** learn layered parameters, commonly with gradient optimization; **LLMs** are large neural sequence models trained on tokens. None of these model families is the entire application around it.


In [ ]:
from collections import Counter, defaultdict
from copy import deepcopy
from hashlib import sha256
from math import log, sqrt
import json, platform, sys

TRAIN=[
 {"id":1,"priority":"normal","text":"reset password account","label":"account"},
 {"id":2,"priority":"normal","text":"cannot sign in account","label":"account"},
 {"id":3,"priority":"urgent","text":"card charged twice invoice","label":"billing"},
 {"id":4,"priority":"normal","text":"copy of invoice billing","label":"billing"},
 {"id":5,"priority":"normal","text":"app crashes dashboard","label":"technical"},
 {"id":6,"priority":"urgent","text":"upload service error","label":"technical"},]
TEST=[
 {"text":"password sign in","label":"account"},
 {"text":"invoice duplicate charge","label":"billing"},
 {"text":"upload crashes error","label":"technical"},]
DOCS=[
 {"id":"K1","text":"Reset password from account security. Sign in problems may require recovery."},
 {"id":"K2","text":"For duplicate card charge compare invoice identifiers before billing review."},
 {"id":"K3","text":"For upload errors check file size and retry after reopening the application."},]
print("structured fields:",[k for k in TRAIN[0] if k!='text'])
print("unstructured field:",TRAIN[0]['text'])


## Training → model state → inference

**Prediction checkpoint:** before running, predict whether inference changes learned model state. Training below uses labels to create token statistics; inference reads them. Retraining after changing training data creates new model state.


In [ ]:
def tok(s): return [w.strip('.,!?;:').lower() for w in s.split() if w.strip('.,!?;:')]
def train_classifier(rows):
    docs=Counter(); words=defaultdict(Counter); vocab=set()
    for r in rows:
        docs[r['label']]+=1
        for w in tok(r['text']): words[r['label']][w]+=1; vocab.add(w)
    return {'labels':sorted(docs),'docs':dict(docs),'words':{k:dict(words[k]) for k in sorted(docs)},'vocab':sorted(vocab)}
def digest(m): return sha256(json.dumps(m,sort_keys=True).encode()).hexdigest()[:12]
def predict(m,text):
    scores={}; n=sum(m['docs'].values()); V=max(1,len(m['vocab']))
    for label in m['labels']:
        c=m['words'][label]; total=sum(c.values()); s=log(m['docs'][label]/n)
        for w in tok(text): s+=log((c.get(w,0)+1)/(total+V))
        scores[label]=s
    return max(scores,key=scores.get),scores
MODEL=train_classifier(TRAIN); before=digest(MODEL)
for r in TEST: print(r['text'],'->',predict(MODEL,r['text'])[0])
assert digest(MODEL)==before
TRAIN2=deepcopy(TRAIN)+[{"id":7,"priority":"normal","text":"refund duplicate charge invoice","label":"billing"}]
MODEL2=train_classifier(TRAIN2)
print('inference digest unchanged:',before,'retrained digest:',digest(MODEL2))
assert digest(MODEL2)!=before


## Embeddings, retrieval, and RAG

An **embedding** is a numerical representation. This transparent toy embedding is lexical; production embeddings are often neural. **Retrieval** ranks external information at run time. **RAG** supplies retrieved context to a generator during inference; retrieval can change context without changing generator weights.

**Prediction checkpoint:** predict the top document for each query and whether retrieval changes `MODEL`. The deterministic template below is not an LLM; it only exposes the RAG-style data path.


In [ ]:
VOCAB=sorted({w for d in DOCS for w in tok(d['text'])})
def embed(text):
    c=Counter(tok(text)); return [float(c.get(w,0)) for w in VOCAB]
def cosine(a,b):
    dot=sum(x*y for x,y in zip(a,b)); na=sqrt(sum(x*x for x in a)); nb=sqrt(sum(y*y for y in b))
    return 0.0 if not na or not nb else dot/(na*nb)
DV={d['id']:embed(d['text']) for d in DOCS}
def retrieve(q,k=1):
    qv=embed(q); rows=[dict(d,score=cosine(qv,DV[d['id']])) for d in DOCS]
    return sorted(rows,key=lambda r:(-r['score'],r['id']))[:k]
def rag_style(q):
    d=retrieve(q)[0]; return {'doc':d['id'],'score':round(d['score'],3),'context':d['text'],'generator':'template, not LLM'}
rd=digest(MODEL)
for q in ['duplicate card charge invoice','password account recovery','upload file error']: print(q,'=>',rag_style(q))
assert digest(MODEL)==rd


## Tools, agents/controllers, memory, evaluation, observability

A **tool** performs an external action/computation. A controller/agent owns **control flow**. **Memory** preserves application/session state. Evaluation measures behavior; observability records traces/latency/failures. Metrics may inform a later training loop but are not themselves training.

**Prediction checkpoint:** predict which request will call the escalation tool and what application state will mutate.


In [ ]:
def escalation_tool(ticket_id,reason): return {'tool':'escalation','ticket':ticket_id,'reason':reason,'queued':True}
def run_application(r,model,memory):
    label,scores=predict(model,r['text']); trace=[('inference',label)]
    doc=retrieve(r['text'])[0]; trace.append(('retrieval',doc['id']))
    tool=None
    if r['priority']=='urgent': tool=escalation_tool(r['id'],label); trace.append(('tool',tool))
    memory.update(runs=memory.get('runs',0)+1,last_intent=label,last_ticket=r['id']); trace.append(('memory',dict(memory)))
    return {'label':label,'doc':doc['id'],'tool':tool,'trace':trace}
MEMORY={}
print(run_application({'id':201,'priority':'normal','text':'reset account password'},MODEL,MEMORY))
print(run_application({'id':202,'priority':'urgent','text':'duplicate invoice charge'},MODEL,MEMORY))
def evaluate(model,rows):
    out=[predict(model,r['text'])[0]==r['label'] for r in rows]; return {'accuracy':sum(out)/len(out),'correct':out}
print('evaluation:',evaluate(MODEL,TEST))
print('runtime:',{'compute':'single CPU process','platform':platform.system(),'python':sys.version.split()[0],'network_calls':0})


## Inspect the whole toy system

Inputs/outputs show **data flow**; the conditional tool call shows **control flow**. Compute/infrastructure hosts every layer even when, as here, it is just one CPU process and in-memory state.


In [ ]:
SYSTEM_MAP=[
 ('TRAIN','data','records','labelled examples','stored examples'),
 ('train_classifier','training','labelled examples','MODEL','creates learned state'),
 ('MODEL','model state','training output','token statistics','read by inference'),
 ('predict','inference','MODEL + new text','label + scores','does not mutate MODEL'),
 ('embed/retrieve','retrieval','query + index','ranked docs','external index state'),
 ('escalation_tool','tool','ticket + reason','action result','external action'),
 ('run_application','application/controller','request + outputs','response/trace','owns control flow'),
 ('MEMORY','memory','run result','state for later run','mutates app state'),
 ('evaluate/trace','evaluation/observability','expected + observed','metrics/events','records evidence'),
 ('runtime','compute/infrastructure','configuration','execution environment','hosts components')]
for c,l,i,o,s in SYSTEM_MAP: print(f'{l:24} {c:18} {i} -> {o} | {s}')


## Controlled failure — diagnose before revealing

Faulty diagram: `TRAINING: query embedding → MODEL WEIGHTS: vector database → INFERENCE: retrieval → MODEL: controller chooses calculator → TRAINING: calculator → MEMORY: evaluation dashboard`.

Before the next cell, identify at least four conflations using inputs, outputs, state mutation, and control flow.


In [ ]:
repair={
 'query embedding':'run-time representation/inference unless parameters are trained',
 'vector database':'external retrieval/index state, not model weights',
 'document retrieval':'retrieval',
 'controller':'application/agent control flow',
 'calculator':'tool execution',
 'evaluation dashboard':'evaluation/observability, not conversation memory'}
for k,v in repair.items(): print(k,'=>',v)
print('Do not claim online weight updates without evidence of an objective, optimizer/training job, and new model state.')


## No-AI gate and transfer

Close the notebook and complete `missions/M01/no_ai_gate.md` from memory. Your map must locate training, inference, retrieval, tools, memory, evaluation/observability, and infrastructure, with separate data-flow and control-flow arrows.

Then complete `missions/M01/assessment.yaml`. The unseen scenarios test transfer, including the ability to say **“no training step is described”** instead of inventing one.
